In [0]:
# Try a dedicated catalog; Free Edition may only allow the built-in 'workspace' catalog
try:
    spark.sql("CREATE CATALOG IF NOT EXISTS power_prices COMMENT 'Project 5: European electricity prices (ENTSO-E)'")
    CATALOG = "power_prices"
except Exception as e:
    print("Could not create catalog, falling back to 'workspace':", str(e)[:200])
    CATALOG = "workspace"

schemas = {
    "bronze": "Raw ENTSO-E API responses as received, plus load metadata",
    "silver": "Parsed, typed, UTC-normalized hourly data, deduplicated",
    "gold":   "Star schema (dimensions and facts) and dashboard aggregates",
    "ops":    "Run log, load watermarks and data quality results",
}
for name, comment in schemas.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{name} COMMENT '{comment}'")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.bronze.raw_files COMMENT 'Raw XML files, one per API call'")

# Remove the temporary setup volume (already empty)
spark.sql("DROP VOLUME IF EXISTS workspace.default.setup")

print(f"Catalog in use: {CATALOG}")
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))
display(spark.sql(f"SHOW VOLUMES IN {CATALOG}.bronze"))